# Notebook de Construccion de Dataset

Este cuaderno documenta paso a paso la construccion de los datasets analiticos para el proyecto de IA predictiva de matricula universitaria.

Objetivo: dejar trazabilidad tecnica y metodologica de las decisiones de datos previas al entrenamiento de modelos.

## 1. Contexto del problema

El presente dataset se va a crear a partir de la información contenida en la base de datos del sistema informático que se utiliza para la administración de la Universidad del Norte Santo Tomás de Aquino (UNSTA). Este sistema cuenta con diversas funcionalidades, entre ellas la de guardar la información básica de las personas que aspiran a cursar una carrera en la institución.

En este proyecto se pretende sacar provecho de esa información, acumulada a través de los años, implementando modelos de inteligencia artificial predictiva. El objetivo es predecir la cantidad de estudiantes que se matricularán en el próximo cuatrimestre en dos niveles:

1. **Total de la universidad.**
2. **Distribución aproximada por carrera.**

La información operativa proviene de tablas transaccionales (`aspirante`, `alumno`, `persona`) que se consolidan en vistas analíticas para el modelado.

### Contenido de las tablas mencionadas

Estas tablas contienen la información que brinda cada persona al ingresar al portal del "Sistema de Preinscripción UNSTA". El proceso funciona de la siguiente manera:

* **Tabla `aspirante`**: Se guardan los datos de la carrera que la persona aspira a cursar.
* **Tabla `persona`**: Se almacenan los datos personales (nombre, contacto, etc.) que el interesado decide brindar.
* **Tabla `alumno`**: Se registran los datos académicos relevantes para el funcionamiento del sistema. Esta tabla es clave para el desarrollo del proyecto, ya que permite confirmar si un aspirante finalmente se convirtió en un alumno de una carrera de la UNSTA.


## 2. Reglas de negocio adoptadas

1. Un aspirante se considera matriculado si `id_alumno IS NOT NULL` en la tabla aspirante.
2. Unidad temporal oficial: `periodo` (anio) + `id_cuatrimestre`.
3. Los registros con `id_cuatrimestre = 0` se expanden a cuatrimestre 1 y 2.
4. Se evita doble conteo por persona-anio-cuatrimestre.
5. Ventana historica recomendada para entrenamiento: desde 2007.

## Instalación de dependencias

**Ejecutá esta celda una sola vez** (la primera vez que abras el notebook o al cambiar de entorno). Luego reiniciá el kernel y ejecutá el resto de celdas.

In [1]:
# Instalar todas las librerías necesarias para este notebook y el EDA (test.ipynb).
# Ejecutar una vez; después: Kernel -> Restart y correr el resto.
%pip install pandas sqlalchemy pymysql boto3 numpy matplotlib seaborn python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

## 3. Conexion a base MySQL

Las credenciales se cargan desde el archivo `.env` en la raíz del proyecto. Asegurate de tener definidas:
- `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DB`
- Opcional: `MYSQL_CONNECT_TIMEOUT` (segundos; por defecto 15)

**Si aparece *TimeoutError* o *Can't connect to MySQL server*:** el host no es alcanzable desde tu red (firewall, VPN, o el servidor solo acepta conexiones desde otra red). Comprobá VPN, que el puerto 3306 esté abierto y que MySQL escuche en esa IP.

In [2]:
from pathlib import Path
from dotenv import load_dotenv

# Cargar variables desde .env (raíz del proyecto: carpeta padre de notebook_example)
_env_path = Path.cwd().parent / '.env'
if _env_path.exists():
    load_dotenv(_env_path)
else:
    load_dotenv()  # fallback: busca .env en el directorio actual

MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = int(os.getenv('MYSQL_PORT', '3306'))
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', '')
MYSQL_DB = os.getenv('MYSQL_DB', 'eUNSTAv3')
MYSQL_CONNECT_TIMEOUT = int(os.getenv('MYSQL_CONNECT_TIMEOUT', '15'))  # segundos

conn_str = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}?connect_timeout={MYSQL_CONNECT_TIMEOUT}'
engine = create_engine(conn_str)
print(f'Conectado a {MYSQL_DB} en {MYSQL_HOST}:{MYSQL_PORT} con {MYSQL_USER}')

Conectado a eUNSTAv3 en 10.172.0.12:3306 con arancelario_user


## 4. Exploracion inicial de volumen historico

Este control permite verificar la distribucion de aspirantes por anio y justificar el corte de periodos con baja densidad de registros.

In [3]:
q_hist = """
SELECT periodo AS anio, COUNT(*) AS aspirantes
FROM aspirante
WHERE periodo IS NOT NULL
GROUP BY periodo
ORDER BY periodo;
"""

df_hist = pd.read_sql(text(q_hist), engine)
df_hist

,anio,aspirantes
0,0,5
1,1980,1
2,1986,1
3,1996,2
4,2000,33
5,2001,54
6,2002,37
7,2003,33
8,2004,55
9,2005,93


## 5. Proceso de construcción de las vistas analíticas

### 5.1 Punto de partida: las tablas transaccionales

El sistema operativo de la UNSTA registra la actividad de preinscripción en tres tablas principales:

| Tabla | Rol en el proceso |
|---|---|
| `aspirante` | Registra la carrera a la que aspira cada persona y el período. Clave: `id_alumno` para saber si se matriculó. |
| `persona` | Datos personales del interesado (nombre, contacto, etc.). |
| `alumno` | Datos académicos del alumno efectivamente matriculado. |

### 5.2 Problemas detectados en los datos crudos

Al analizar las tablas transaccionales se identificaron los siguientes problemas que motivaron la creación de vistas intermedias:

1. **Doble conteo por persona**: una misma persona puede aparecer varias veces en `aspirante` para el mismo período (reinscripciones, correcciones administrativas). Sin quitar duplicados, el total de aspirantes queda sobreestimado.
2. **`id_cuatrimestre = 0`**: algunos registros históricos tienen cuatrimestre `0`, lo que significa que el dato no fue informado. Bajo la regla de negocio adoptada, estos registros se expanden a cuatrimestre `1` y `2` para no perder historia.
3. **`fecha_efectivizacion` no confiable**: se descartó como eje temporal porque presenta inconsistencias históricas. En su lugar se usa `periodo` (año) + `id_cuatrimestre` como unidad temporal estable.
4. **Registros previos a 2007**: el volumen y la calidad de los datos antes de 2007 son insuficientes para entrenamiento. Se define 2007 como límite inferior de la ventana de análisis.

### 5.3 Diseño de las vistas SQL

A partir de esos problemas y las reglas de negocio definidas en la sección 2, se construyeron tres vistas en MySQL:

**Paso 1 — `vw_aspirante_base`**  
Normaliza y quita duplicados de la tabla `aspirante`, combinando `periodo`, `id_cuatrimestre` y `id_persona` como clave única. Expande los registros con `id_cuatrimestre = 0` en dos filas (cuatrimestres 1 y 2). El campo `matriculado` se deriva directamente de si `id_alumno IS NOT NULL`.

**Paso 2 — `vw_dm_matricula_total`**  
Agrega `vw_aspirante_base` por `periodo` e `id_cuatrimestre`, sumando la cantidad de personas con `matriculado = 1`. Genera la serie temporal de matriculados a nivel universidad completa.

**Paso 3 — `vw_dm_matricula_carrera`**  
Agrega `vw_aspirante_base` por `periodo`, `id_cuatrimestre` e `id_carrera`, sumando los matriculados por carrera. Genera una serie temporal por carrera para el modelo de nivel desagregado.

### 5.4 Scripts SQL de creación de las vistas

A continuación se documentan los scripts ejecutados directamente en MySQL para crear las tres vistas analíticas. Se incluyen como referencia técnica y para garantizar la reproducibilidad del proceso.

---

#### Vista 1 — `vw_aspirante_base`

Normaliza y deduplica `aspirante`. Expande los registros con `id_cuatrimestre = 0` en dos filas (cuatrimestres 1 y 2).

```sql
DROP VIEW IF EXISTS vw_aspirante_base;

CREATE VIEW vw_aspirante_base AS

  -- Registros con cuatrimestre informado (1 o 2)
  SELECT
    a.id_aspirante,
    a.id_persona,
    a.id_alumno,
    a.periodo AS anio,
    a.id_cuatrimestre AS cuatrimestre
  FROM aspirante a
  WHERE a.periodo IS NOT NULL
    AND a.id_cuatrimestre IN (1, 2)

  UNION ALL

  -- Registros con cuatrimestre = 0: se expanden al cuatrimestre 1
  SELECT
    a.id_aspirante,
    a.id_persona,
    a.id_alumno,
    a.periodo AS anio,
    1 AS cuatrimestre
  FROM aspirante a
  WHERE a.periodo IS NOT NULL
    AND a.id_cuatrimestre = 0

  UNION ALL

  -- Registros con cuatrimestre = 0: se expanden al cuatrimestre 2
  SELECT
    a.id_aspirante,
    a.id_persona,
    a.id_alumno,
    a.periodo AS anio,
    2 AS cuatrimestre
  FROM aspirante a
  WHERE a.periodo IS NOT NULL
    AND a.id_cuatrimestre = 0;
```

---

#### Vista 2 — `vw_dm_matricula_total`

Agrega por año y cuatrimestre el total de aspirantes y matriculados a nivel universidad. La columna objetivo del modelo es `matriculados_objetivo`.

```sql
DROP VIEW IF EXISTS eUNSTAv3.vw_dm_matricula_total;

CREATE VIEW eUNSTAv3.vw_dm_matricula_total AS
SELECT
  b.anio,
  b.cuatrimestre,
  COUNT(*)                                          AS aspirantes_total,
  SUM(CASE WHEN b.id_alumno IS NOT NULL THEN 1
           ELSE 0 END)                              AS aspirantes_con_id_alumno,
  COUNT(DISTINCT
    CASE
      WHEN b.id_alumno IS NOT NULL
        THEN CONCAT(b.id_persona, '-', b.anio, '-', b.cuatrimestre)
      ELSE NULL
    END
  )                                                 AS matriculados_objetivo
FROM eUNSTAv3.vw_aspirante_base b
WHERE b.anio BETWEEN 2007 AND 2025
GROUP BY b.anio, b.cuatrimestre;
```

> **Nota:** `matriculados_objetivo` usa `COUNT(DISTINCT CONCAT(...))` para evitar el doble conteo de una misma persona matriculada en el mismo período.

---

#### Vista 3 — `vw_dm_matricula_carrera`

Agrega por año, cuatrimestre y carrera. Resuelve primero la deduplicación por persona-período y luego hace el join con la tabla `alumno` para obtener `id_carrera`. Se enriquece con **`oferta_educativa`** para incluir el nombre de la carrera (`nombre`, `abreviatura`, `nombre_largo`), de modo que los CSV exportados a MinIO y el EDA puedan mostrar etiquetas legibles en los gráficos.

```sql
DROP VIEW IF EXISTS eUNSTAv3.vw_dm_matricula_carrera;

CREATE VIEW eUNSTAv3.vw_dm_matricula_carrera AS
SELECT
  base.anio,
  base.cuatrimestre,
  base.id_carrera,
  base.matriculados_carrera,
  TRIM(oe.nombre)       AS nombre_carrera,
  TRIM(oe.abreviatura)  AS abreviatura_carrera,
  TRIM(oe.nombre_largo) AS nombre_largo_carrera
FROM (
  SELECT
    d.anio,
    d.cuatrimestre,
    COALESCE(al.id_carrera, 'SIN_CARRERA') AS id_carrera,
    COUNT(*)                               AS matriculados_carrera
  FROM (
    SELECT
      b.anio,
      b.cuatrimestre,
      b.id_persona,
      MIN(b.id_alumno) AS id_alumno
    FROM eUNSTAv3.vw_aspirante_base b
    WHERE b.anio BETWEEN 2007 AND 2025
      AND b.id_alumno IS NOT NULL
    GROUP BY b.anio, b.cuatrimestre, b.id_persona
  ) d
  LEFT JOIN eUNSTAv3.alumno al ON al.id_alumno = d.id_alumno
  GROUP BY d.anio, d.cuatrimestre, COALESCE(al.id_carrera, 'SIN_CARRERA')
) base
LEFT JOIN eUNSTAv3.oferta_educativa oe
  ON TRIM(CAST(base.id_carrera AS CHAR)) = TRIM(oe.id_carrera);
```

> **Nota:** El `LEFT JOIN` con `oferta_educativa` permite que registros como `SIN_CARRERA` o códigos no existentes en la oferta sigan apareciendo; las columnas `nombre_carrera`, `abreviatura_carrera` y `nombre_largo_carrera` quedarán en NULL en esos casos.

---

### 5.5 Vistas resultantes

Las vistas fueron creadas en la base de datos `eUNSTAv3` para estandarizar la capa analítica:
- `vw_aspirante_base`: base normalizada y sin duplicados por persona-año-cuatrimestre.
- `vw_dm_matricula_total`: dataset final agregado por año-cuatrimestre (nivel universidad).
- `vw_dm_matricula_carrera`: dataset final agregado por año-cuatrimestre-carrera **con nombre y abreviatura de carrera** (para exportación a MinIO y EDA).

In [4]:
df_base_sample = pd.read_sql(text('SELECT * FROM vw_aspirante_base LIMIT 10;'), engine)
df_base_sample

,id_aspirante,id_persona,id_alumno,anio,cuatrimestre
0,431,864,None,2007,1
1,450,6566,None,2009,1
2,486,7091,None,2006,1
3,490,7093,None,2006,1
4,492,7094,None,2006,1
5,494,7095,None,2006,1
6,495,7104,None,2006,1
7,496,7105,None,2006,1
8,501,1001421,None,2006,1
9,502,7112,None,2006,1


## 6. Dataset final: total universidad

Variable objetivo principal: `matriculados_objetivo` por (`anio`, `cuatrimestre`).

In [5]:
q_total = """
SELECT *
FROM vw_dm_matricula_total
WHERE anio >= 2007
ORDER BY anio, cuatrimestre;
"""

df_total = pd.read_sql(text(q_total), engine)
print(df_total.shape)
df_total.head()

(38, 5)


,anio,cuatrimestre,aspirantes_total,aspirantes_con_id_alumno,matriculados_objetivo
0,2007,1,2074,1.0,1
1,2007,2,246,0.0,0
2,2008,1,1900,0.0,0
3,2008,2,154,0.0,0
4,2009,1,1752,0.0,0


In [6]:
# Control de completitud temporal (2 cuatrimestres por anio esperado)
semestres_por_anio = df_total.groupby('anio')['cuatrimestre'].nunique().reset_index(name='n_cuatrimestres')
semestres_por_anio

,anio,n_cuatrimestres
0,2007,2
1,2008,2
2,2009,2
3,2010,2
4,2011,2
5,2012,2
6,2013,2
7,2014,2
8,2015,2
9,2016,2


## 7. Dataset final: matrícula por carrera

### 7.1 Propósito del dataset desagregado

El modelo de predicción del proyecto opera en **dos niveles de granularidad**:

| Nivel | Variable objetivo | Fuente |
|---|---|---|
| Universidad (total) | `matriculados_objetivo` | `vw_dm_matricula_total` |
| Por carrera (desagregado) | `matriculados_carrera` | `vw_dm_matricula_carrera` |

Este segundo dataset permite entrenar un modelo independiente por carrera o bien un modelo multi-serie que capture patrones propios de cada programa académico (estacionalidad, tendencia, caídas por discontinuación de carreras, etc.).

La clave de observación es la terna `(anio, cuatrimestre, id_carrera)`, que identifica unívocamente la cantidad de alumnos matriculados en una carrera particular para un período dado.

**Enriquecimiento con `oferta_educativa`:** la vista **`vw_dm_matricula_carrera`** (ver script en sección 5.4) ya incluye un **LEFT JOIN** con la tabla **`oferta_educativa`** sobre `id_carrera`. Con ello el dataset trae de origen las columnas que permiten **conocer el nombre de la carrera**:

- **`nombre_carrera`** — nombre corto (campo `nombre` de `oferta_educativa`).
- **`nombre_largo_carrera`** — nombre completo o descriptivo (`nombre_largo`).
- **`abreviatura_carrera`** — sigla o abreviatura (`abreviatura`).

Así se puede interpretar cada `id_carrera` en reportes, EDA y exportaciones sin tener que cruzar con otra tabla. En `oferta_educativa`, `id_carrera` es `varchar(2)`; para un matching seguro se usa `TRIM(CAST(v.id_carrera AS CHAR)) = TRIM(oe.id_carrera)`. Los registros sin match (por ejemplo `SIN_CARRERA` o códigos inexistentes en la oferta) conservan NULL en esas columnas.

### 7.2 Verificación de consistencia entre niveles

Una condición necesaria —aunque no suficiente— de calidad de datos es que la **suma de matriculados por carrera sea igual al total de matriculados a nivel universidad** para cada período.

Esta verificación es importante por dos razones:

1. **Integridad referencial**: si las dos vistas derivaron correctamente de la misma fuente (`vw_aspirante_base`) y aplicaron la misma deduplicación, la suma agregada debe coincidir. Una discrepancia indicaría un error en la lógica SQL de alguna de las dos vistas.
2. **Trazabilidad experimental**: en proyectos de machine learning sobre datos reales, las inconsistencias entre niveles de agregación son una fuente frecuente de sesgo. Documentar este chequeo explicitamente en el notebook constituye evidencia de rigor metodológico ante la evaluación del proyecto.

El resultado esperado es `diff = 0` en todos los períodos. Cualquier diferencia positiva o negativa debe ser investigada antes de iniciar el entrenamiento de modelos.

In [7]:
q_carrera = """
SELECT *
FROM vw_dm_matricula_carrera
WHERE anio >= 2007
ORDER BY anio, cuatrimestre, id_carrera;
"""

df_carrera = pd.read_sql(text(q_carrera), engine)
print(df_carrera.shape)
df_carrera.head()

(1011, 4)


,anio,cuatrimestre,id_carrera,matriculados_carrera
0,2007,1,29,1
1,2013,1,18,1
2,2013,1,49,4
3,2013,2,12,1
4,2014,1,01,43


In [8]:
# Paso 1: agregar matriculados por carrera a nivel período (suma sobre todas las carreras)
chk = df_carrera.groupby(['anio', 'cuatrimestre'], as_index=False)['matriculados_carrera'].sum()

# Paso 2: unir con el total universidad para comparar ambas fuentes en la misma fila
chk = chk.merge(
    df_total[['anio', 'cuatrimestre', 'matriculados_objetivo']],
    on=['anio', 'cuatrimestre'],
    how='left'
)

# Paso 3: calcular la diferencia (debe ser 0 en todos los períodos)
chk['diff'] = chk['matriculados_carrera'] - chk['matriculados_objetivo']

# Paso 4: aserción automática — falla si existe alguna inconsistencia
inconsistencias = chk[chk['diff'] != 0]
assert inconsistencias.empty, (
    f'ALERTA: {len(inconsistencias)} período(s) con inconsistencia entre niveles:\n{inconsistencias}'
)
print(f'Verificación OK: diff = 0 en los {len(chk)} períodos analizados.')

chk.head(20)

,anio,cuatrimestre,matriculados_carrera,matriculados_objetivo,diff
0,2007,1,1,1,0
1,2013,1,5,5,0
2,2013,2,1,1,0
3,2014,1,1185,1185,0
4,2014,2,112,112,0
5,2015,1,1395,1395,0
6,2015,2,95,95,0
7,2016,1,1243,1243,0
8,2016,2,86,86,0
9,2017,1,1833,1833,0


## 8. Export de datasets para entrenamiento

Se exportan snapshots versionados para trazabilidad experimental.

In [ ]:
out_dir = Path('../training/data')
out_dir.mkdir(parents=True, exist_ok=True)

df_total.to_csv(out_dir / 'dataset_matricula_total.csv', index=False)
df_carrera.to_csv(out_dir / 'dataset_matricula_carrera.csv', index=False)

print('Archivos exportados en:', out_dir.resolve())

## 9. Cierre metodologico

Con estos datasets consolidados se puede iniciar la etapa de modelado y validacion temporal (baseline, modelos de series temporales y registro en MLflow).

Este notebook debe mantenerse como evidencia de decisiones de datos para el informe final.